# QSAR Random Forest BorutaShap feature selection

This notebook performs BorutaShap feature selection for Random Forest across all targets. It saves per-target checkpoints and generates Z-score based summary plots.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

MODEL_DIR = "random-forest"
MODEL_NAME = "Random Forest"

cwd = Path.cwd()
project_root = Path("..").resolve() if cwd.name == MODEL_DIR else cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from qsar_config import DATA_PATH, RANDOM_SEED
from qsar_common import (
    add_mol_column,
    aggregate_targets_by_fingerprint,
    build_morgan_fingerprints,
    encode_targets,
    load_qsar_dataset,
    stack_features_and_targets,
)

In [2]:
BORUTA_TRIALS = 20

In [3]:
df = load_qsar_dataset(DATA_PATH)

In [4]:
df = add_mol_column(df, smiles_column="Smiles", mol_column="mol")

In [5]:
FPSIZE = 4096
df = build_morgan_fingerprints(
    df,
    mol_column="mol",
    output_column="fp",
    radius=2,
    n_bits=FPSIZE,
)

In [6]:
df, encoder, target_names = encode_targets(
    df,
    target_column="Target",
    output_column="target_encoded",
)

In [7]:
df_agg = aggregate_targets_by_fingerprint(
    df,
    fp_column="fp",
    encoded_target_column="target_encoded",
    aggregated_target_column="target",
)

In [8]:
x, y = stack_features_and_targets(
    df_agg,
    fp_column="fp",
    target_column="target",
)

In [9]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.3,
    random_state=RANDOM_SEED,
)

In [ ]:
import numpy as np
import scipy.stats
import shap

if not hasattr(np, 'NaN'):
    np.NaN = np.nan
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'int'):
    np.int = int

if not hasattr(scipy.stats, 'binom_test'):
    def binom_test_wrapper(x, n=None, p=0.5, alternative='two-sided'):
        k_clean = int(x)
        n_clean = int(n) if n is not None else None

        return scipy.stats.binomtest(k=k_clean, n=n_clean, p=p, alternative=alternative).pvalue

    scipy.stats.binom_test = binom_test_wrapper

def _shap_values_to_feature_matrix(values, n_samples, n_features):
    if hasattr(values, 'values'):
        values = values.values

    if isinstance(values, list):
        arr = np.asarray(values)
        if arr.ndim == 3:
            if arr.shape[1] == n_samples and arr.shape[2] == n_features:
                return np.asarray(arr[-1], dtype=np.float32)
            if arr.shape[0] == n_samples and arr.shape[1] == n_features:
                return np.asarray(arr[:, :, -1], dtype=np.float32)
        raise ValueError(f'Unsupported SHAP list shape: {arr.shape}')

    arr = np.asarray(values)
    if arr.ndim == 2 and arr.shape == (n_samples, n_features):
        return arr.astype(np.float32)
    if arr.ndim == 3:
        if arr.shape[0] == n_samples and arr.shape[1] == n_features:
            return arr[:, :, -1].astype(np.float32)
        if arr.shape[1] == n_samples and arr.shape[2] == n_features:
            return arr[-1].astype(np.float32)
        if arr.shape[0] == n_features and arr.shape[1] == n_samples:
            return arr[:, :, -1].T.astype(np.float32)
    raise ValueError(f'Unsupported SHAP array shape: {arr.shape}')

from BorutaShap import BorutaShap

def _patched_explain(self):
    explainer = shap.TreeExplainer(
        self.model,
        feature_perturbation='tree_path_dependent',
        approximate=True,
    )
    data = self.find_sample() if self.sample else self.X_boruta
    values = explainer.shap_values(data, check_additivity=False)
    arr = _shap_values_to_feature_matrix(values, n_samples=data.shape[0], n_features=data.shape[1])
    self.shap_values = np.abs(arr).mean(axis=0)

BorutaShap.explain = _patched_explain
import pandas as pd

feature_names = [f"Bit_{i}" for i in range(x.shape[1])]
X_df = pd.DataFrame(x, columns=feature_names)

target_names = encoder.categories_[0]

print(f"Feature Matrix Shape: {X_df.shape}")
print(f"Targets to analyze: {target_names}")

Feature Matrix Shape: (7207, 4096)
Targets to analyze: ['ar' 'era' 'erb' 'gr' 'mr' 'pr']


In [11]:
import numpy as np

for i, name in enumerate(target_names):
    n_pos = y[:, i].sum()
    n_neg = y.shape[0] - n_pos
    print(f"{name}: positives={n_pos}, negatives={n_neg}")

ar: positives=1802, negatives=5405
era: positives=2425, negatives=4782
erb: positives=1501, negatives=5706
gr: positives=1630, negatives=5577
mr: positives=781, negatives=6426
pr: positives=1368, negatives=5839


In [12]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from BorutaShap import BorutaShap

repo_root = Path.cwd().resolve()
if repo_root.name == "random-forest":
    repo_root = repo_root.parent.parent
elif repo_root.name == "qsar":
    repo_root = repo_root.parent

qsar_dir = repo_root / "qsar"
if str(qsar_dir) not in sys.path:
    sys.path.insert(0, str(qsar_dir))

from checkpoint_utils import file_signature, load_pickle_cache, save_pickle_cache

CACHE_VERSION = 1
cache_dir = repo_root / "qsar" / "random-forest" / "checkpoints" / "boruta-shap"
cache_dir.mkdir(parents=True, exist_ok=True)
plot_dir = Path("images") / "boruta-shap" / "random-forest"
plot_dir.mkdir(parents=True, exist_ok=True)

feature_names = [f"bit_{i}" for i in range(x.shape[1])]
X_boruta = pd.DataFrame(x, columns=feature_names)

selected_features_per_target = {}
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=7,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

zscore_results = []
data_path = (qsar_dir / "nr_ic_merged.csv").resolve()
data_sig = file_signature(data_path)
PER_TARGET_TOP_N = 20

for i, target_name in enumerate(target_names):
    print(f"\n--- Processing target: {target_name} ---")
    y_target = pd.Series(y[:, i], name=str(target_name))

    safe_name = str(target_name).replace("/", "_").replace(" ", "_")
    boruta_cache_path = cache_dir / f"target_{safe_name}_scores.pkl"
    boruta_cache_meta = {
        "version": CACHE_VERSION,
        "kind": "qsar_boruta_target_scores",
        "target_name": str(target_name),
        "data": data_sig,
        "n_bits": int(x.shape[1]),
        "random_seed": int(RANDOM_SEED),
        "boruta_n_trials": int(BORUTA_TRIALS),
        "model": {
            "type": "RandomForestClassifier",
            "n_estimators": int(getattr(rf_model, "n_estimators", 0)),
            "max_depth": int(getattr(rf_model, "max_depth", 0) or 0),
            "class_weight": str(getattr(rf_model, "class_weight", None)),
        },
    }

    cached = load_pickle_cache(boruta_cache_path, boruta_cache_meta)

    if cached is None:
        selector = BorutaShap(
            model=rf_model,
            importance_measure="shap",
            classification=True,
        )

        try:
            selector.fit(
                X=X_boruta,
                y=y_target,
                n_trials=BORUTA_TRIALS,
                sample=False,
                verbose=True,
            )

            mean_z_scores = selector.history_x.mean(axis=0)
            mean_z_scores.name = str(target_name)

            selector.TentativeRoughFix()
            subset = selector.Subset()
            selected_features = subset.columns.tolist()

            save_pickle_cache(
                boruta_cache_path,
                boruta_cache_meta,
                {
                    "mean_z_scores": mean_z_scores,
                    "selected_features": selected_features,
                    "accepted": [str(v) for v in getattr(selector, "accepted", [])],
                    "tentative": [str(v) for v in getattr(selector, "tentative", [])],
                    "rejected": [str(v) for v in getattr(selector, "rejected", [])],
                },
            )
            cache_status = "miss"

        except Exception as e:
            print(f"    -> Failed for {target_name}: {e}")
            continue
    else:
        mean_z_scores = cached["mean_z_scores"]
        selected_features = cached.get("selected_features", [])
        cache_status = "hit"

    zscore_results.append(mean_z_scores)
    selected_features_per_target[target_name] = list(selected_features)

    target_abs = mean_z_scores.abs().sort_values(ascending=False)
    target_top = target_abs.head(PER_TARGET_TOP_N)
    if len(target_top) > 0:
        plt.figure(figsize=(10, max(4, 0.35 * len(target_top))))
        plt.barh(range(len(target_top)), target_top.values[::-1])
        plt.yticks(range(len(target_top)), target_top.index[::-1])
        plt.xlabel("Mean |Z-score|")
        plt.title(f"BorutaShap top bits: {target_name}")
        plt.tight_layout()
        target_plot_path = plot_dir / f"boruta_plot_{safe_name}.svg"
        plt.savefig(target_plot_path, format="svg")
        plt.close()

    print(f"    -> cache={cache_status} | selected features={len(selected_features)}")

final_zscore_df = pd.DataFrame(zscore_results).fillna(0)
print("Z-score matrix prepared in memory.")


--- Processing target: ar ---
    -> cache=hit | selected features=239

--- Processing target: era ---
    -> cache=hit | selected features=264

--- Processing target: erb ---
    -> cache=hit | selected features=206

--- Processing target: gr ---
    -> cache=hit | selected features=263

--- Processing target: mr ---
    -> cache=hit | selected features=197

--- Processing target: pr ---
    -> cache=hit | selected features=199
Z-score matrix prepared in memory.


In [13]:
TOP_N = 20
z_abs = final_zscore_df.abs()
mean_abs = z_abs.mean(axis=0).sort_values(ascending=False)

if len(mean_abs) == 0:
    raise RuntimeError("Nebyla nalezena žádná Z-skóre pro vykreslení.")

top_bits = mean_abs.head(TOP_N).index.tolist()
plot_df = final_zscore_df[top_bits]

plt.figure(figsize=(max(10, TOP_N * 0.45), max(6, 0.35 * len(plot_df))))
row_order = plot_df.abs().sum(axis=1).sort_values(ascending=False).index
plot_df_sorted = plot_df.loc[row_order]

im = plt.imshow(plot_df_sorted.values, aspect="auto", cmap="coolwarm")
plt.colorbar(im, label="Průměrné Z-skóre (BorutaShap)")
plt.yticks(np.arange(plot_df_sorted.shape[0]), plot_df_sorted.index)
plt.xticks(np.arange(len(top_bits)), top_bits, rotation=90)
plt.title(f"Nejlepších {TOP_N} bitů podle průměru |Z| napříč cíli (BorutaShap)")
plt.tight_layout()
heatmap_path = plot_dir / f"boruta_top_{TOP_N}_zscore_heatmap.svg"
plt.savefig(heatmap_path, format="svg")
plt.close()

plt.figure(figsize=(max(10, TOP_N * 0.5), 5))
plt.bar(range(len(top_bits)), mean_abs.loc[top_bits].values)
plt.xticks(range(len(top_bits)), top_bits, rotation=90)
plt.ylabel("Průměr(|Z|) napříč cíli")
plt.title(f"Nejlepších {TOP_N} bitů podle průměru |Z| (BorutaShap)")
plt.tight_layout()
bar_path = plot_dir / f"boruta_top_{TOP_N}_zscore_bar.svg"
plt.savefig(bar_path, format="svg")
plt.close()

print(f"Saved top-N plots: {heatmap_path.name}, {bar_path.name}")

Saved top-N plots: boruta_top_20_zscore_heatmap.svg, boruta_top_20_zscore_bar.svg
